# 🌿 Seasonal NDVI Mosaic Creation

## Purpose

This notebook creates annual seasonal NDVI mosaics from previously generated NDVI rasters.

The workflow:

- Reads NDVI raster files
- Extracts acquisition dates from filenames
- Groups rasters by year and season
- Creates annual spring and summer mosaics
- Exports seasonal composite rasters

---

## Input

Expected input filename format:

```text
YYYY-MM-DD_NDVI.tif
```

Example:

```text
2020-05-14_NDVI.tif
```

---

## Seasons Used

```text
Spring = May–June
Summer = August–September
```

---

## Output

Example outputs:

```text
NDVI_spring_2020.tif
NDVI_summer_2020.tif
```


## 1️⃣ Import Libraries

Import ArcPy, OS utilities, and datetime functions required for raster processing and seasonal grouping.


In [ ]:
import arcpy
import os
from datetime import datetime

# Allow overwriting existing outputs
arcpy.env.overwriteOutput = True

## 2️⃣ Define Input and Output Folders

Specify the folder containing NDVI rasters and the output folder for seasonal mosaics.


In [ ]:
# Update these paths before running the notebook
root_folder = r"path\to\input\NDVI_clean"
output_folder = r"path\to\output\NDVI_seasonal_mosaic"

# Create output folder if it does not exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Dictionary for storing rasters grouped by year and season
data = {}

## 3️⃣ Group Rasters by Year and Season

This step:

- Searches for NDVI raster files
- Extracts acquisition dates from filenames
- Groups rasters into spring and summer categories
- Organizes rasters by year


In [ ]:
# --------------------------------------------------
# Read rasters and group them by year and season
# --------------------------------------------------

for subdir, dirs, files in os.walk(root_folder):
    for f in files:

        # Process only NDVI raster files
        if "NDVI" in f and f.lower().endswith((".tif", ".tiff")):

            full_path = os.path.join(subdir, f)

            try:
                # Extract date from filename
                date_str = f.split("_NDVI")[0]
                date = datetime.strptime(date_str, "%Y-%m-%d")

                year = date.year
                month = date.month

                # Create dictionary structure if year does not exist
                if year not in data:
                    data[year] = {
                        "spring": [],
                        "summer": []
                    }

                # Spring: May–June
                if month in [5, 6]:
                    data[year]["spring"].append(full_path)

                # Summer: August–September
                elif month in [8, 9]:
                    data[year]["summer"].append(full_path)

            except:
                print(f"Could not process file: {f}")

## 4️⃣ Create Annual Seasonal Mosaics

For each year:

- Create one spring mosaic
- Create one summer mosaic
- Export the mosaics as GeoTIFF rasters

ArcPy tool used:

```text
MosaicToNewRaster
```


In [ ]:
# --------------------------------------------------
# Create annual seasonal mosaics
# --------------------------------------------------

for year in sorted(data.keys()):

    print(f"\n=== {year} ===")

    spring_list = data[year]["spring"]
    summer_list = data[year]["summer"]

    print("Spring rasters:", len(spring_list))
    print("Summer rasters:", len(summer_list))

    # Create spring mosaic
    if len(spring_list) > 0:

        arcpy.management.MosaicToNewRaster(
            input_rasters=spring_list,
            output_location=output_folder,
            raster_dataset_name_with_extension=f"NDVI_spring_{year}.tif",
            coordinate_system_for_the_raster="",
            pixel_type="32_BIT_FLOAT",
            number_of_bands=1,
            mosaic_method="MAXIMUM",
            mosaic_colormap_mode="FIRST"
        )

        print(f"Spring mosaic created: {year}")

    # Create summer mosaic
    if len(summer_list) > 0:

        arcpy.management.MosaicToNewRaster(
            input_rasters=summer_list,
            output_location=output_folder,
            raster_dataset_name_with_extension=f"NDVI_summer_{year}.tif",
            coordinate_system_for_the_raster="",
            pixel_type="32_BIT_FLOAT",
            number_of_bands=1,
            mosaic_method="MAXIMUM",
            mosaic_colormap_mode="FIRST"
        )

        print(f"Summer mosaic created: {year}")

print("\nSeasonal mosaic generation completed.")